# RAG Mini Project: phishing triage runbook
Parts 1 to 5 are the practice pipeline (chunk, embed and store, retrieve, augment, score). Part 6 runs it on my own documents in my_kb/ and my own questions in goldset.json.

In [ ]:
import json, os, glob, re
import pandas as pd
import matplotlib.pyplot as plt

# keep this on the CPU so no GPU setup problems
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

from sentence_transformers import SentenceTransformer
import chromadb

# my own documents and questions
COURSE_KB = sorted(glob.glob("my_kb/0*.md"))
GOLDSET_FILE = "goldset.json"

# ONE embedding model for both the chunks and the questions
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(texts, normalize_embeddings=True).tolist()

print(COURSE_KB)

## Part 1: chunk the documents (TODO 1)

In [ ]:
def split_into_chunks(text, chunk_size=600, overlap=100):
    pieces = []
    start = 0
    while start < len(text):
        piece = text[start:start + chunk_size].strip()
        if piece:
            pieces.append(piece)
        start = start + (chunk_size - overlap)
    return pieces


def load_and_chunk(files, chunk_size=600, overlap=100):
    chunks, sources = [], []
    for path in files:
        text = open(path, encoding="utf-8").read()
        pieces = split_into_chunks(text, chunk_size, overlap)
        chunks += pieces
        sources += [os.path.basename(path)] * len(pieces)
        print("  %-32s -> %2d chunks" % (os.path.basename(path), len(pieces)))
    return chunks, sources

chunks, sources = load_and_chunk(COURSE_KB)
print("  TOTAL:", len(chunks), "chunks")
print("First chunk starts:", chunks[0][:70])

## Part 2: embed and store in ChromaDB (TODO 2)

In [ ]:
client = chromadb.EphemeralClient()
_counter = [0]

def build_collection(chunks, sources):
    _counter[0] += 1
    name = "kb_%d" % _counter[0]
    collection = client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})
    collection.add(ids=["c%d" % i for i in range(len(chunks))],
                   documents=chunks,
                   embeddings=embed(chunks),
                   metadatas=[{"source": s} for s in sources])
    return collection

collection = build_collection(chunks, sources)
print("Stored", collection.count(), "chunks")

## Part 3: retrieve (TODO 3)

In [ ]:
def retrieve(collection, question, k=4, max_distance=0.75):
    results = collection.query(query_embeddings=embed([question]), n_results=k)
    hits = []
    for doc, dist, meta in zip(results["documents"][0],
                               results["distances"][0],
                               results["metadatas"][0]):
        if dist <= max_distance:
            hits.append({"text": doc, "distance": dist, "source": meta["source"]})
    return hits

for q in ["How quickly must an employee report a suspicious email?",
          "What is the capital of France?"]:
    found = retrieve(collection, q)
    print("Q:", q)
    print("   hits:", len(found))
    for h in found:
        print("     distance %.3f  [%s]  %s..." % (h["distance"], h["source"],
                                                   h["text"][:60].replace("\n", " ")))
    print()

## Part 4: augment the prompt and generate (TODO 4)

In [ ]:
SYSTEM = (
    "You are a cybersecurity assistant. Answer the question using ONLY the context below.\n"
    "Cite the source file for every claim, like [01_reporting.md]. If the answer is in the context, start "
    "your reply with [RAG]. If the context is empty or does not contain the answer, say you do not "
    "know and start with [LLM]. Do not answer from memory.\n\n# Context\n{context}"
)

def build_context(hits):
    if not hits:
        return ""
    blocks = []
    for i, h in enumerate(hits, start=1):
        blocks.append("[%d] (source: %s)\n%s" % (i, h["source"], h["text"]))
    return "\n\n".join(blocks)


_USE_API = [True]

def generate(system, user):
    if os.getenv("OPENAI_API_KEY") and _USE_API[0]:
        try:
            from openai import OpenAI
            r = OpenAI().chat.completions.create(
                model="gpt-4o-mini", temperature=0,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}])
            return r.choices[0].message.content
        except Exception as e:
            _USE_API[0] = False
            print("   (LLM unavailable, offline mode: %s)" % type(e).__name__)
    context = system.split("# Context\n", 1)[-1]
    if context.strip() == "" or context.strip() == "(nothing found)":
        return "[LLM] I don't know. The documents do not cover this. (offline mode)"
    return "[RAG] (offline mode) From the retrieved context:\n" + context[:300]


def ask(collection, question, k=4, max_distance=0.75, show=True):
    hits = retrieve(collection, question, k, max_distance)
    context = build_context(hits)
    system = SYSTEM.format(context=context if context else "(nothing found)")
    answer = generate(system, question)
    if show:
        print("Q:", question)
        print("   grounded?", "YES, %d chunk(s)" % len(hits) if hits else "NO, nothing retrieved")
        print("   answer:", answer[:300].replace("\n", "\n           "), "\n")
    return {"hits": hits, "answer": answer}

_ = ask(collection, "What does Severity 1 mean?")
_ = ask(collection, "How should analysts respond to a ransomware infection?")

## Part 5: measure it (TODO 5)

In [ ]:
GOLDSET = json.load(open(GOLDSET_FILE))["questions"]
print("Loaded", len(GOLDSET), "questions\n")

def found_evidence(hits, evidence):
    for h in hits:
        for phrase in evidence:
            if phrase.lower() in h["text"].lower():
                return True
    return False


def score(collection, goldset, k=4, max_distance=0.75):
    found, refused = 0, 0
    for q in goldset:
        hits = retrieve(collection, q["question"], k, max_distance)
        if q["answerable"]:
            if found_evidence(hits, q["evidence"]):
                found += 1
        else:
            if len(hits) == 0:
                refused += 1
    return found, refused


n_answerable = sum(q["answerable"] for q in GOLDSET)
n_unanswerable = len(GOLDSET) - n_answerable
found, refused = score(collection, GOLDSET)
print("found   %d/%d answerable questions" % (found, n_answerable))
print("refused %d/%d unanswerable questions" % (refused, n_unanswerable))

## Part 6: my data. Load and check

In [ ]:
paths = sorted(glob.glob("my_kb/0*.md"))
texts = {os.path.basename(p): open(p, encoding="utf-8").read() for p in paths}
print(len(texts), "documents,", sum(len(t) for t in texts.values()), "characters")

MY_GOLD = json.load(open("goldset.json"))["questions"]
norm = lambda s: re.sub(r"\s+", " ", s).lower()
alltext = norm(" ".join(texts.values()))
for q in MY_GOLD:
    for ph in q["evidence"]:
        print(norm(ph) in alltext, "|", q["qid"], "|", ph)

## Build my RAG and ask my questions (transcript)

In [ ]:
my_files = [os.path.join("my_kb", n) for n in texts]
my_chunks, my_sources = load_and_chunk(my_files)
my_col = build_collection(my_chunks, my_sources)
print("Stored", my_col.count(), "chunks\n")

for q in MY_GOLD:
    _ = ask(my_col, q["question"])

## Best distance per question (helps pick the threshold)

In [ ]:
for q in MY_GOLD:
    r = my_col.query(query_embeddings=embed([q["question"]]), n_results=1)
    print("%.3f" % r["distances"][0][0], "ANSWERABLE" if q["answerable"] else "REFUSE", "|", q["question"])

## Experiments (one setting at a time)

In [ ]:
def run(chunk_size=600, overlap=100, k=4, max_distance=0.75):
    ch, src = [], []
    for path in my_files:
        pieces = split_into_chunks(open(path, encoding="utf-8").read(), chunk_size, overlap)
        ch += pieces
        src += [os.path.basename(path)] * len(pieces)
    col = build_collection(ch, src)
    f, r = score(col, MY_GOLD, k, max_distance)
    return dict(chunk_size=chunk_size, overlap=overlap, k=k,
                max_distance=max_distance, num_chunks=len(ch),
                found=f, refused=r)

base = dict(chunk_size=600, overlap=100, k=4, max_distance=0.75)
configs = [base]
for s in (300, 1200):
    configs.append({**base, "chunk_size": s, "overlap": s // 6})
for k in (1, 2, 8):
    configs.append({**base, "k": k})
for d in (0.45, 0.55, 0.65, 0.85):
    configs.append({**base, "max_distance": d})

df = pd.DataFrame([run(**c) for c in configs])
df.to_csv("results.csv", index=False)
df

## Chart

In [ ]:
labels = ["s%d/k%d/d%.2f" % (r.chunk_size, r.k, r.max_distance) for r in df.itertuples()]
x = list(range(len(df)))
plt.figure(figsize=(11, 4))
plt.bar([i - 0.2 for i in x], df.found, width=0.4, label="found (of 6)")
plt.bar([i + 0.2 for i in x], df.refused, width=0.4, label="refused (of 2)")
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylabel("questions")
plt.title("Found vs refused by configuration")
plt.legend()
plt.tight_layout()
plt.savefig("figure.png", dpi=150)
plt.show()